<a href="https://colab.research.google.com/github/heberdavi/mba-engsoft-tcc/blob/main/notebooks/01-processamento_pln.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

📑 Guia de Execução Estratégica
⚠️ IMPORTANTE: Sempre que o Runtime (Ambiente de Execução) for reiniciado, as Células 1 e 2 devem ser executadas obrigatoriamente para restabelecer os caminhos do Drive e reinstalar as bibliotecas.

🔄 Fluxo de Dependências:
Sessão Recém-Iniciada: Executar Célula 1 ➔ Célula 2.

Primeira vez no projeto: Executar Célula 1 ➔ Célula 2 ➔ Célula 3 (Carga).

Retomando Processamento: Se o banco já existe no Drive, pule a Célula 3 e vá direto para a Célula 4 e/ou 5 e/ou 6.

In [ ]:
# Célula 1: Montagem do Google Drive e Configuração de Caminhos
from google.colab import drive
import os

# 1. Montagem Segura: Só executa se ainda não estiver montado
if not os.path.exists('/content/drive/MyDrive'):
    print("📂 Montando Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive já está montado e acessível.")

# 2. Configuração Estrita de Caminhos
DRIVE_DIR = '/content/drive/MyDrive/mba-engsof-tcc/versao_final'
DB_FILE_NAME = 'data/base-dados.db'
DB_PATH = os.path.join(DRIVE_DIR, DB_FILE_NAME)

# Artefatos SQL
SCHEMA_SQL = os.path.join(DRIVE_DIR, 'sql/01-schema.sql')
SEED_SQL = os.path.join(DRIVE_DIR, 'sql/02-seed_data.sql')

# Pasta de Saída (Outputs)
EXPORT_PATH = os.path.join(DRIVE_DIR, 'outputs')
if not os.path.exists(EXPORT_PATH):
    os.makedirs(EXPORT_PATH)
    print(f"📁 Pasta de exportação criada em: {EXPORT_PATH}")

print(f"📍 Banco de Dados: {DB_PATH}")

In [ ]:
# Célula 2: Instalação das bibliotecas e inicialização da estrutura (Schema)

# 1. Instalação Silenciosa
!pip install -q transformers torch pandas bertopic pysentimiento spacy
!python -m spacy download pt_core_news_lg -q

import sqlite3
import torch

# 2. Hardware Check para BERTimbau/BERTopic
device = 0 if torch.cuda.is_available() else -1

def inicializar_estrutura_db(db_path, schema_path):
    """Garante que a estrutura de tabelas esteja presente."""
    print(f"🛠️ Verificando integridade das tabelas...")

    # Se o arquivo de banco não existir, o SQLite o criará automaticamente
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        with open(schema_path, 'r', encoding='utf-8') as f:
            cursor.executescript(f.read())
        conn.commit()
        print("✅ Estrutura (Schema) validada com sucesso!")
    except Exception as e:
        print(f"❌ Erro ao processar Schema: {e}")
    finally:
        conn.close()

# 3. Execução
inicializar_estrutura_db(DB_PATH, SCHEMA_SQL)

print(f"\n🚀 Ambiente pronto (GPU: {'Ativa' if device == 0 else 'Inativa'}).")

In [ ]:
# Célula 3: Carga Inicial de Dados (Seed SQL)
def executar_carga_dados(db_path, seed_path):
    """Popula o banco apenas se a tabela 'verso' estiver vazia."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        # Verifica se já existem dados para evitar duplicidade no Drive
        cursor.execute("SELECT count(*) FROM verso")
        total_existente = cursor.fetchone()[0]

        if total_existente > 0:
            print(f"ℹ️ O banco já contém {total_existente} versos. Carga inicial ignorada.")
            return

        print("🌱 Semeando dados iniciais (02-seed_data.sql)... Isso pode levar alguns minutos.")
        with open(seed_path, 'r', encoding='utf-8') as f:
            cursor.executescript(f.read())

        conn.commit()
        print(f"✅ Carga de {seed_path} concluída com sucesso!")

    except sqlite3.OperationalError as e:
        print(f"⚠️ Erro operacional: {e}. Verifique se a Célula 2 foi executada.")
    except Exception as e:
        print(f"❌ Erro crítico na carga: {e}")
    finally:
        conn.close()

# Executa a carga (Somente se necessário)
executar_carga_dados(DB_PATH, SEED_SQL)

In [ ]:
# Célula 4: Limpeza Estrutural e Filtro de Densidade (Antidoto ao Ruído Nominal)
import spacy
import sqlite3
import pandas as pd
import re

# Carrega o modelo de português
try:
    nlp = spacy.load("pt_core_news_lg")
except:
    !python -m spacy download pt_core_news_lg
    nlp = spacy.load("pt_core_news_lg")

def limpar_texto_estrutural(texto):
    if not texto or len(texto.strip()) < 3: return "RUIDO_CURTO"

    doc = nlp(texto)

    # Filtramos tokens válidos (substantivos, verbos, adjetivos e nomes próprios)
    # Ignoramos stop words e pontuação
    tokens = [t for t in doc if not t.is_stop and not t.is_punct and t.pos_ in ['NOUN', 'VERB', 'ADJ', 'PROPN']]

    if not tokens: return "RUIDO_VAZIO"

    # Métrica 1: Densidade de Nomes Próprios (PROPN)
    # Se mais de 70% do conteúdo significativo forem nomes próprios, é provavelmente uma lista/genealogia
    propn_count = len([t for t in tokens if t.pos_ == 'PROPN'])
    propn_ratio = propn_count / len(tokens)

    # Métrica 2: Presença de Ação/Estado
    # Antídotos existenciais raramente são frases sem verbos ou adjetivos
    has_action_or_state = any(t.pos_ in ['VERB', 'ADJ'] for t in tokens)

    # CASO CRÍTICO (Ex: Nesias e Hatifa):
    # Verso curto, alta proporção de nomes próprios e sem verbo
    if len(tokens) <= 3 and propn_ratio > 0.5 and not has_action_or_state:
        return "RUIDO_NOMINAL"

    # Retornamos o texto limpo (usando o texto original para preservar a semântica)
    return " ".join([t.text.lower() for t in tokens])

# Execução e Persistência
conn = sqlite3.connect(DB_PATH)
df_versos = pd.read_sql_query("SELECT id, texto FROM verso", conn)

print("🧼 Limpando textos e aplicando filtros de densidade gramatical...")
df_versos['texto_limpo'] = df_versos['texto'].apply(limpar_texto_estrutural)

cursor = conn.cursor()
cursor.execute("DELETE FROM verso_limpo")
df_versos[['id', 'texto_limpo']].rename(columns={'id': 'verso_id'}).to_sql(
    'verso_limpo',
    conn,
    if_exists='append', # Importante: 'append' preserva a estrutura e índices
    index=False
)

conn.commit()
conn.close()
print("✅ Célula 4 concluída! Ruídos nominais e estruturais pré-identificados.")

In [ ]:
# Célula 5: Classificação por Eixos Existenciais
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from transformers import pipeline
import sqlite3
import pandas as pd
import numpy as np
import os
from tqdm.notebook import tqdm

# 1. Configuração e Carga de Dados
conn = sqlite3.connect(DB_PATH)
query = """
    SELECT v.id as verso_id, v.texto, l.abreviacao, g.id as genero_id, vl.texto_limpo
    FROM verso v
    JOIN verso_limpo vl ON v.id = vl.verso_id
    JOIN livro l ON l.id = v.livro_id
    JOIN genero_literario g ON g.id = l.genero_id
"""
df_input = pd.read_sql_query(query, conn)

# --- TRATAMENTO DE ERRO DE TIPO (ANTI-TYPEERROR) ---
df_input['texto_limpo'] = df_input['texto_limpo'].fillna('vazio').astype(str)
docs_para_classificar = [str(doc).strip() if (doc and str(doc).strip() != "") else "vazio" for doc in df_input['texto_limpo'].tolist()]

print(f"🔍 Verificação de tipos na lista: {set([type(d) for d in docs_para_classificar])}")
# --------------------------------------------------

# 2. Inicialização do Modelo BERTimbau (Zero-Shot)
embedding_model = pipeline("feature-extraction", model="neuralmind/bert-base-portuguese-cased", device=0)

# Descrições Refinadas para a Linguagem NVI e Análise Teoterapêutica
descricoes_eixos = [
    # Eixo 0: Exaustão vs. Refrigério
    "Estados de Fadiga Existencial e Alívio da Alma: Descreve o peso da existência, o estar psicologicamente sobrecarregado, o desânimo e o abatimento espiritual. Inclui sentimentos de angústia profunda e o clamor por socorro. Em contrapartida, oferece o refrigério da paz interior, o descanso para a alma atribulada, o consolo nas aflições e o alívio emocional que torna o fardo da vida leve e suave.",

    # Eixo 1: Transitoriedade vs. Solidez
    "Fragilidade Humana e Firmeza Espiritual: Aborda a brevidade da vida, a impermanência dos dias e a insegurança das coisas mundanas que perecem como a erva. Em oposição, destaca a Rocha Eterna, o fundamento espiritual inabalável, a confiança inabalável em Deus e a segurança de quem constrói a identidade sobre valores eternos, imutáveis e constantes, independentemente das circunstâncias externas.",

    # Eixo 2: Vazio vs. Propósito
    "Crise de Sentido e Vocação Existencial: O sentimento de que tudo é vaidade e a futilidade de uma vida sem significado. Trata do vácuo existencial e da desorientação mental. Contrastando com isso, apresenta a descoberta de um sentido para a vida, o chamado vocacional, os planos de esperança futura, o valor intrínseco do indivíduo e a compreensão de uma missão ou razão de ser que transcende a existência biológica ou material.",

    # Eixo 3: Narrativo/Normativo (Controle)
    "Registros Narrativos, Leis e Informações Factuais: Conteúdo puramente informativo, administrativo ou instrutivo. Inclui genealogias, listas de nomes, medidas técnicas, rituais, censos e relatos de viagens. Abrange fundamentalmente fórmulas de introdução de diálogos e marcadores de transição narrativa como 'disse', 'respondeu', 'falou', 'perguntou', servindo como uma categoria técnica para textos sem carga emocional ou existencial direta."
]

model_topic = BERTopic(
    embedding_model=embedding_model,
    zeroshot_topic_list=descricoes_eixos,
    zeroshot_min_similarity=0.1,
    calculate_probabilities=True,
    vectorizer_model=CountVectorizer(ngram_range=(1, 2))
)

print("🤖 Classificando via Zero-Shot com âncoras conceituais calibradas para NVI...")
topics, probs_matrix = model_topic.fit_transform(docs_para_classificar)

# 3. Lógica de Decisão com Threshold Dinâmico e Margem de Dominância
final_topics = []
final_probs = []

print("⚖️ Aplicando Filtragem por Gênero Literário...")

for i, row in tqdm(df_input.iterrows(), total=len(df_input), desc="Processando Bíblia", mininterval=0):

    p_exaustao, p_transitoriedade, p_vazio, p_narrativo = probs_matrix[i][0:4]
    existenciais = [p_exaustao, p_transitoriedade, p_vazio]
    best_idx = np.argmax(existenciais)
    best_score = existenciais[best_idx]

    # Tratamento Implícito: Se o modelo está em dúvida entre Sentimento e Narrativa,
    # a margem será pequena.
    margem_dominancia = best_score - p_narrativo
    genero_id = row['genero_id']

    # --- DECISÃO POR GÊNERO ---
    if genero_id in [1, 2]: # Pentateuco e Histórico: Rigor Máximo
        threshold = 0.88
        if best_score > threshold and margem_dominancia > 0.15:
            final_topics.append(best_idx)
            final_probs.append(best_score)
        else:
            final_topics.append(3)
            final_probs.append(p_narrativo)

    elif genero_id == 3: # Poético/Sapiencial: Alta Sensibilidade
        threshold = 0.52
        if best_score > threshold:
            final_topics.append(best_idx)
            final_probs.append(best_score)
        else:
            final_topics.append(3)
            final_probs.append(p_narrativo)

    elif genero_id == 6: # Epístola
        threshold = 0.65
        if best_score > threshold and best_score > p_narrativo:
            final_topics.append(best_idx)
            final_probs.append(best_score)
        else:
            final_topics.append(3)
            final_probs.append(p_narrativo)

    else: # Profético (4), Evangelho (5) e Apocalíptico (7)
        threshold = 0.75
        if best_score > threshold and best_score > p_narrativo:
            final_topics.append(best_idx)
            final_probs.append(best_score)
        else:
            final_topics.append(3)
            final_probs.append(p_narrativo)

# 4. Persistência no SQLite
print("💾 Atualizando banco de dados...")
mapa_eixos = {0: "Exaustão vs. Refrigério", 1: "Transitoriedade vs. Solidez", 2: "Vazio vs. Propósito", 3: "Narrativo/Normativo"}

df_verso_topico = pd.DataFrame({'verso_id': df_input['verso_id'], 'topico_id': final_topics, 'similaridade': final_probs})

cursor = conn.cursor()
cursor.execute("DELETE FROM verso_topico")
cursor.execute("DELETE FROM topico")

df_mapa = pd.DataFrame([{'id': k, 'antidoto_referencia': v} for k, v in mapa_eixos.items()])
df_mapa.to_sql('topico', conn, if_exists='append', index=False)
df_verso_topico.to_sql('verso_topico', conn, if_exists='append', index=False)

conn.commit()
conn.close()
print(f"✨ Célula 5 concluída! {len(df_verso_topico)} versículos processados.")

In [ ]:
# Célula 6: Análise de Sentimento Contextual e Cruzamento Existencial
from pysentimiento import create_analyzer
import pandas as pd
import sqlite3
from tqdm.auto import tqdm

# 1. Inicializar o Analisador
print("🚀 Carregando modelo Transformer para Sentimento (PT-BR)...")
# O analisador 'sentiment' para 'pt' é baseado em BERTimbau, ideal para o TCC
analyzer = create_analyzer(task="sentiment", lang="pt")

# 2. Busca do texto original e dos tópicos
conn = sqlite3.connect(DB_PATH)
df_input = pd.read_sql_query("""
    SELECT v.id as verso_id, v.texto, vt.topico_id
    FROM verso v
    JOIN verso_topico vt ON v.id = vt.verso_id
""", conn)

textos = df_input['texto'].tolist()
verso_ids = df_input['verso_id'].tolist()

# 3. Execução da análise em lotes (Aproveitando a GPU se disponível)
print(f"📊 Analisando carga emocional de {len(textos)} versículos...")
sentimentos = []
batch_size = 64
mapa_num = {'POS': 1, 'NEU': 0, 'NEG': -1}

# O predict em lote é significativamente mais rápido no Colab
for i in tqdm(range(0, len(textos), batch_size)):
    lote = textos[i:i + batch_size]
    ids_lote = verso_ids[i:i + batch_size]
    preds_lote = analyzer.predict(lote)

    for idx, p in enumerate(preds_lote):
        # Capturamos as probabilidades brutas para análises de incerteza se necessário
        sentimentos.append({
            'verso_id': ids_lote[idx],
            'label': p.output,
            'sentimento_num': mapa_num.get(p.output, 0),
            'score_pos': p.probas.get('POS', 0),
            'score_neg': p.probas.get('NEG', 0),
            'score_neu': p.probas.get('NEU', 0)
        })

df_sent = pd.DataFrame(sentimentos)

# 4. Persistência dos Resultados
try:
    cursor = conn.cursor()
    # Limpamos para garantir que a nova classificação da Célula 5 seja a única presente
    cursor.execute("DELETE FROM verso_sentimento")

    # Inserimos os novos resultados (Integridade referencial com 'verso_id')
    df_sent.to_sql('verso_sentimento', conn, if_exists='append', index=False)
    conn.commit()
    print("\n✅ Célula 6 concluída! Sentimentos processados e salvos com sucesso.")

    # 5. RESULTADO FINAL: O DIAGNÓSTICO (PROBLEMA) VS. A CURA (ANTÍDOTO)
    print("\n📈 RESUMO EXECUTIVO: PROBLEMÁTICA (CRISE) VS. ANTÍDOTO (CURA)")

    res_final = pd.read_sql_query("""
        SELECT
            t.antidoto_referencia as Eixo_Filosofico,
            COUNT(*) as Total_Versos,
            SUM(CASE WHEN vs.sentimento_num = 1 THEN 1 ELSE 0 END) as Antidotos_Cura,
            SUM(CASE WHEN vs.sentimento_num = -1 THEN 1 ELSE 0 END) as Problematica_Crise,
            ROUND(AVG(vs.sentimento_num), 3) as Polaridade_Media
        FROM verso_topico vt
        JOIN topico t ON vt.topico_id = t.id
        JOIN verso_sentimento vs ON vt.verso_id = vs.verso_id
        WHERE t.id != 3 -- Foco nos eixos Han, Bauman e Frankl
        GROUP BY t.antidoto_referencia
        ORDER BY Polaridade_Media DESC
    """, conn)

    # Exibe a tabela formatada no Colab
    display(res_final)

except Exception as e:
    print(f"❌ Erro na persistência: {e}")
finally:
    conn.close()